# Field sample explorer

Rotatable surface, height map and line cut for one TMF8829 capture log, scrubbable across the
measurement sets.

```
cd python-poc
uv sync
uv run jupyter lab explore_capture.ipynb
```

Point it at a different log by editing `CAPTURE` in the next cell and re-running the notebook.

In [ ]:
# CAPTURE = "field-tests/evm-test-silo-cheio-00-out_UID004f845e_20260812-120958.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-cheio-01-centro_UID004f845e_20260812-121020.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-cheio-02-centro_UID004f845e_20260812-121045.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-cheio-03-parede_UID004f845e_20260812-121102.ndjson.gz"
CAPTURE = "field-tests/evm-test-silo-cheio-04-girando_UID004f845e_20260812-121121.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-vazio-01_UID004f845e_20260812-120214.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-vazio-02-inclinado_UID004f845e_20260812-120235.ndjson.gz"
# CAPTURE = "field-tests/evm-test-silo-vazio-03-girando_UID004f845e_20260812-120346.ndjson.gz"

PEAK = 0  # which of the four peak slots to reconstruct from; changeable in the viewer too

## Loading

Re-derives every value from the raw ZeroMQ payload in each record. The decoded copy stored
alongside it is a convenience, not the record of truth, so this reparses like `replay.py` does and
stays correct if the vendor parsers change.

In [ ]:
import base64
import datetime as dt
import os
from dataclasses import dataclass

import numpy as np

import capture_log

NO_TARGET = 0
_FP_GRID = {0: (8, 8), 1: (8, 8), 2: (16, 16), 3: (32, 32), 4: (32, 32)}


@dataclass
class Capture:
    """One capture log as arrays indexed [set, y, x] (and [set, y, x, peak] where per-peak)."""

    source: str
    header: dict
    sets: list
    dist: np.ndarray
    signal: np.ndarray
    snr: np.ndarray
    noise: np.ndarray
    xtalk: np.ndarray
    dx: np.ndarray
    dy: np.ndarray
    zcorr: np.ndarray
    unit: str

    @property
    def config(self):
        return self.header.get("config", {})

    @property
    def label(self):
        return self.header.get("label") or os.path.basename(self.source)

    def axial(self, peak=0):
        return self.dist[..., peak] / self.zcorr

    def seconds(self):
        stamps = [s["t"] for s in self.sets if s.get("t")]
        if len(stamps) < 2:
            return None
        parse = lambda v: dt.datetime.strptime(v, "%Y-%m-%dT%H:%M:%S.%fZ")
        return (parse(stamps[-1]) - parse(stamps[0])).total_seconds()


def _directions(fp_mode):
    # Reproduces Tmf8829AppCommon.zCorrection. Inlined rather than imported: it is four lines of
    # arithmetic, and reaching for it would pull the vendor application layer into the notebook.
    cols, rows = _FP_GRID.get(fp_mode)
    x = (np.arange(cols) - cols / 2 + 0.5) / (cols * 3.0 / 4.0)
    y = (np.arange(rows) - rows / 2 + 0.5) / rows
    dx, dy = np.meshgrid(x, y)
    return dx, dy, np.sqrt(1 + dx * dx + dy * dy)


def load(path):
    header, records = None, []
    for record in capture_log.read_log(path):
        if record.get("type") == "header":
            header = record
        elif record.get("type") == "set":
            records.append(record)
    if header is None:
        raise ValueError("{}: no header record; not a capture log".format(path))

    config = header.get("config", {})
    # `select` is what distinguishes distances from bin indices; decode_set hands the conversion
    # itself to Tmf8829AppCommon, so peaks arrive already in mm.
    to_mm = config["select"] >= 1
    unit = "mm" if to_mm else "bins"
    dx, dy, zcorr = _directions(config["fp_mode"])
    rows, cols = zcorr.shape
    n = len(records)

    dist = np.full((n, rows, cols, 4), np.nan)
    signal = np.full((n, rows, cols, 4), np.nan)
    snr = np.full((n, rows, cols, 4), np.nan)
    noise = np.full((n, rows, cols), np.nan)
    xtalk = np.full((n, rows, cols), np.nan)
    sets = []

    for i, record in enumerate(records):
        meta = {"seq": record.get("seq"), "t": record.get("host_utc"), "error": None}
        sets.append(meta)
        try:
            decoded = capture_log.decode_set(base64.b64decode(record["raw"]), to_mm=to_mm)
        except Exception as exc:
            meta["error"] = "{}: {}".format(type(exc).__name__, exc)
            continue

        head = (decoded.get("frames") or [{}])[0].get("header", {})
        meta.update({
            "frame": head.get("fNumber"),
            "temp": (head.get("temperature") or [None, None, None])[2],
            "warn": capture_log.frame_warnings(decoded),
            "ok": bool(decoded.get("integrity_ok")),
        })

        pixels = decoded.get("pixels")
        if not pixels:
            meta["error"] = "no result frame"
            continue
        for y, row in enumerate(pixels):
            for x, pixel in enumerate(row):
                noise[i, y, x] = pixel.get("noise")
                xtalk[i, y, x] = pixel.get("xtalk")
                for p, peak in enumerate(pixel["peaks"]):
                    # A missing target stays NaN rather than becoming zero, so gaps read as gaps
                    # instead of as a surface at the sensor.
                    if peak.get("distance") in (None, NO_TARGET):
                        continue
                    dist[i, y, x, p] = peak["distance"]
                    signal[i, y, x, p] = peak.get("signal")
                    snr[i, y, x, p] = peak.get("snr")

    return Capture(source=path, header=header, sets=sets, dist=dist, signal=signal, snr=snr,
                   noise=noise, xtalk=xtalk, dx=dx, dy=dy, zcorr=zcorr, unit=unit)


cap = load(CAPTURE)
cfg, dev = cap.config, cap.header.get("device", {})
secs = cap.seconds()
peak0 = cap.dist[..., PEAK]
hit = int(np.isfinite(peak0).sum())
broken = [s for s in cap.sets if s.get("error") or s.get("ok") is False]
flagged = [s for s in cap.sets if s.get("warn")]

print("{}  |  {} sets{}".format(
    cap.label, len(cap.sets),
    " in {:.1f} s ({:.1f} sets/s)".format(secs, len(cap.sets) / secs) if secs else ""))
print("device 0x{:08x}  fw {}  EVM {}  |  {}".format(
    dev.get("deviceSerialNumber", "?"), ".".join(str(v) for v in dev.get("fwVersion", "?")),
    dev.get("evmVersion_ascii", "?"), cap.header.get("preconfig", "")))
print("{}x{} zones, {} peaks, period {} ms, histograms {}, units {}".format(
    *cap.zcorr.shape, cfg.get("nr_peaks"), cfg.get("period"),
    "on" if cfg.get("histograms") else "off", "0.25 mm" if cfg.get("select", 0) >= 1 else "bins"))
print("peak {}: {} of {} zone-sets had a target  |  radial {:.0f} to {:.0f} {}".format(
    PEAK, hit, peak0.size, np.nanmin(peak0), np.nanmax(peak0), cap.unit))
print("integrity: {}".format(
    "all {} sets intact".format(len(cap.sets)) if not broken
    else "{} set(s) incomplete: {}".format(len(broken), [s["seq"] for s in broken])))
if flagged:
    print("warnings on sets {}".format([s["seq"] for s in flagged]))
if not cap.header.get("notes"):
    print("no field notes recorded; the label is the only description of the scene")

## The viewer

Three linked views of the same frame. PyVista renders through VTK and cannot draw into a matplotlib
axes, so the 3D view is its own widget beside the two matplotlib panels; one set of controls drives
all three.

**Play** animates the sets. **row / column** and **index** move the cut. **peak** switches which of
the four peak slots is reconstructed. **mean of capture** replaces the frame with the run's average.

Colour limits and axis limits are fixed across the whole capture rather than refitted per frame — a
scale that re-fits every frame shows changes the measurement never made.

In [ ]:
import ipywidgets as W
import matplotlib.pyplot as plt
import pyvista as pv
from IPython.display import clear_output, display
from matplotlib.colors import LinearSegmentedColormap

pv.set_jupyter_backend("trame")
plt.ioff()

SURFACE, INK, INK2, MUTED, GRIDLINE = "#fbfcfd", "#0d1117", "#4d545d", "#858c95", "#dfe3e9"
SERIES_1, SERIES_2 = "#2a78d6", "#eb6834"
BLUE = LinearSegmentedColormap.from_list("poc_blue", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#256abf", "#184f95", "#0d366b"])

ROWS, COLS = cap.zcorr.shape
NSETS = len(cap.sets)


class View:
    """Reconstruction of one peak slot: axial distances, their spread, and surface points."""

    def __init__(self, peak):
        self.set_peak(peak)

    def set_peak(self, peak):
        self.peak = peak
        self.ax = cap.axial(peak)
        good = np.isfinite(self.ax)
        if not good.any():
            raise ValueError("peak {} reported no target anywhere in this capture".format(peak))
        self.lo, self.hi = float(self.ax[good].min()), float(self.ax[good].max())

        # Counted by hand rather than with nanmean: a zone that never saw a target would otherwise
        # warn on an all-NaN slice on every reload.
        counts = good.sum(axis=0)
        held = np.maximum(counts, 1)
        mean = np.where(good, self.ax, 0).sum(axis=0) / held
        self.mean = np.where(counts > 0, mean, np.nan)
        var = np.where(good, (self.ax - mean) ** 2, 0).sum(axis=0) / held
        self.sd = np.where(counts > 0, np.sqrt(var), np.nan)
        # A never-measured zone still needs somewhere to sit, or the surface folds to the origin.
        self.fill = np.where(counts > 0, mean, self.ax[good].mean())

    def frame(self, index, use_mean):
        return self.mean if use_mean else self.ax[index]

    def xyz(self, index, use_mean):
        z = self.frame(index, use_mean)
        good = np.isfinite(z)
        pos = np.where(good, z, self.fill)
        return cap.dx * pos, cap.dy * pos, -pos, np.where(good, z, np.nan)


view = View(PEAK)
XSPAN = float(max(np.abs(cap.dx).max(), np.abs(cap.dy).max()) * view.hi * 1.08)

x0, y0, z0, s0 = view.xyz(0, False)
grid = pv.StructuredGrid(x0, y0, z0)
grid["axial"] = s0.ravel(order="F")

plotter = pv.Plotter(window_size=(620, 560))
plotter.set_background(SURFACE)
mesh = plotter.add_mesh(grid, scalars="axial", cmap=BLUE, clim=(view.lo, view.hi),
                        nan_color=MUTED, show_edges=True, edge_color=GRIDLINE,
                        scalar_bar_args={"title": "axial " + cap.unit, "color": INK2,
                                         "title_font_size": 13, "label_font_size": 11,
                                         "vertical": True, "position_x": 0.87})
plotter.add_mesh(pv.Sphere(radius=XSPAN * 0.035), color=SERIES_2)  # the sensor, at the origin
plotter.show_grid(color=MUTED, font_size=9, fmt="%.0f", n_xlabels=3, n_ylabels=3, n_zlabels=3,
                  xtitle="x ({})".format(cap.unit), ytitle="y ({})".format(cap.unit),
                  ztitle="depth below sensor ({})".format(cap.unit))
plotter.camera_position = "iso"
viewer = plotter.show(jupyter_backend="trame", return_viewer=True)

fig, (ax_hm, ax_cut) = plt.subplots(2, 1, figsize=(6.0, 7.0), constrained_layout=True)
fig.patch.set_facecolor(SURFACE)
im = ax_hm.imshow(view.mean, cmap=BLUE, vmin=view.lo, vmax=view.hi, origin="upper",
                  interpolation="nearest")
bar = fig.colorbar(im, ax=ax_hm, fraction=0.046, pad=0.03)
bar.set_label("axial " + cap.unit, color=INK2, fontsize=9)
bar.ax.tick_params(colors=MUTED, labelsize=8)
cut_line, = ax_hm.plot([], [], color=SERIES_2, lw=2.5)
ax_hm.set_xticks(range(COLS), ["x{}".format(i) for i in range(COLS)], fontsize=8)
ax_hm.set_yticks(range(ROWS), ["y{}".format(i) for i in range(ROWS)], fontsize=8)
ax_hm.tick_params(colors=MUTED, length=0)
for spine in ax_hm.spines.values():
    spine.set_color(GRIDLINE)
# Detached from pyplot's registry so the inline backend does not also flush a static copy of it
# under the widget at the end of the cell; display(fig) still renders it inside the Output.
plt.close(fig)

out = W.Output()
play = W.Play(min=0, max=max(0, NSETS - 1), interval=220, show_repeat=False)
sset = W.IntSlider(min=0, max=max(0, NSETS - 1), description="set", continuous_update=False)
W.jslink((play, "value"), (sset, "value"))
axis_w = W.ToggleButtons(options=["row", "column"], value="row",
                         style={"button_width": "72px"})
idx_w = W.IntSlider(min=0, max=ROWS - 1, value=ROWS // 2, description="index")
peak_w = W.Dropdown(options=list(range(4)), value=PEAK, description="peak",
                    layout=W.Layout(width="140px"))
mean_w = W.Checkbox(value=False, description="mean of capture", indent=False)


def redraw(*_):
    index, use_mean = sset.value, mean_w.value
    axis, idx = axis_w.value, idx_w.value
    z = view.frame(index, use_mean)

    gx, gy, gz, scalars = view.xyz(index, use_mean)
    grid.points = np.column_stack([gx.ravel(order="F"), gy.ravel(order="F"), gz.ravel(order="F")])
    grid["axial"] = scalars.ravel(order="F")
    plotter.render()

    im.set_data(z)
    if axis == "row":
        cut_line.set_data([-0.5, COLS - 0.5], [idx, idx])
        vals, mu, sd, lat = z[idx, :], view.mean[idx, :], view.sd[idx, :], cap.dx[idx, :]
    else:
        cut_line.set_data([idx, idx], [-0.5, ROWS - 0.5])
        vals, mu, sd, lat = z[:, idx], view.mean[:, idx], view.sd[:, idx], cap.dy[:, idx]

    stamp = ("mean of {} sets".format(NSETS) if use_mean
             else "set {}   {}".format(cap.sets[index]["seq"], cap.sets[index].get("t") or ""))
    ax_hm.set_title("axial distance ({})   {}".format(cap.unit, stamp), color=INK, fontsize=10)

    ax_cut.clear()
    ok = np.isfinite(mu) & np.isfinite(sd)
    if ok.any():
        ax_cut.fill_between((lat * mu)[ok], (mu - sd)[ok], (mu + sd)[ok],
                            color=SERIES_1, alpha=0.16, lw=0,
                            label="capture mean " + r"$\pm$" + " 1 sd")
    ax_cut.plot(lat * np.where(np.isfinite(vals), vals, mu), vals,
                color=SERIES_1, marker="o", ms=4.5, lw=2, label=stamp)
    ax_cut.set_xlim(-XSPAN, XSPAN)
    ax_cut.set_ylim(view.hi * 1.03, view.lo * 0.97)  # inverted: the sensor is above the surface
    ax_cut.set_xlabel("lateral offset from the sensor axis ({})".format(cap.unit),
                      color=INK2, fontsize=9)
    ax_cut.set_ylabel("axial distance ({})".format(cap.unit), color=INK2, fontsize=9)
    ax_cut.set_title("cut along {} {}".format(axis, idx), color=INK, fontsize=10)
    ax_cut.grid(True, color=GRIDLINE, lw=0.8)
    ax_cut.set_axisbelow(True)
    ax_cut.tick_params(colors=MUTED, labelsize=8)
    for spine in ax_cut.spines.values():
        spine.set_color(GRIDLINE)
    legend = ax_cut.legend(fontsize=8, frameon=False, loc="upper right")
    for text in legend.get_texts():
        text.set_color(INK2)

    with out:
        clear_output(wait=True)
        display(fig)


def on_axis(*_):
    idx_w.max = (ROWS if axis_w.value == "row" else COLS) - 1
    redraw()


def on_peak(change):
    view.set_peak(change["new"])
    mesh.mapper.scalar_range = (view.lo, view.hi)
    im.set_clim(view.lo, view.hi)
    redraw()


for widget in (sset, idx_w, mean_w):
    widget.observe(redraw, names="value")
axis_w.observe(on_axis, names="value")
peak_w.observe(on_peak, names="value")

display(W.VBox([
    W.HBox([play, sset, axis_w, idx_w, peak_w, mean_w]),
    W.HBox([viewer, out]),
]))
redraw()

## Conventions and caveats

Zones are indexed `y` down and `x` across, matching the grid printed by `replay.py` and
`capture_8x8_long_range.py`. The sensor sits at the origin of the 3D view with the surface below it,
so the `z` axis there is negated axial distance.

**Axial distance is a model, not a calibration.** It is the reported radial range divided by the
vendor's own per-zone direction cosine (`Tmf8829AppCommon.zCorrection`), which assumes an idealised
field of view. It removes the geometric stretch of the outer zones; it does not correct lens
distortion or mounting tilt.

**Per-zone noise is not uniform across the grid.** In `evm-test-silo-cheio-02-centro` the
set-to-set standard deviation runs 20–35 mm in the central zones and up to 99 mm at the edges, so
edge zones deserve less weight than their numbers suggest. Switch **mean of capture** on and watch
the band in the cut to see it for any capture.

Raw histograms are in the log but are not loaded here — none of these three views use them, and they
are the bulk of a capture's bytes. Re-deriving peak detection offline from them is the obvious next
cell.